In [0]:
# Databricks notebook source
# =============================================================================
# FIU CUSTOMER VELOCITY NETWORK
# Builds a customer-level risk mart for fraud investigators.
# Every customer gets one row with their full activity profile,
# mule indicators, device network risk, and a composite FIU risk score.
# =============================================================================

from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

# ---------------------------------------------------------------------------
# tell Spark where to read from and write to
# ---------------------------------------------------------------------------
catalog = "ubuntu_bank_fraud"
schema_name = "gold"
source_tbl = f"{catalog}.{schema_name}.gold_base_enriched_transactions"
target_tbl = f"{catalog}.{schema_name}.gold_fiu_customer_velocity_network"
batch_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"source: {source_tbl}")
print(f"target: {target_tbl}")
print(f"batch:  {batch_id}")

# load the foundation table we built earlier
base = spark.table(source_tbl)

# quick sanity check — how many rows and unique customers do we have
total_rows = base.count()
unique_custs = base.select("customer_id").distinct().count()
print(f"rows: {total_rows:,}")
print(f"customers: {unique_custs:,}")

source: ubuntu_bank_fraud.gold.gold_base_enriched_transactions
target: ubuntu_bank_fraud.gold.gold_fiu_customer_velocity_network
batch:  20260623_152415
rows: 2,200,000
customers: 28,311


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# before we build anything, check what our transaction types look like
# this helps us understand the data we are aggregating
# ---------------------------------------------------------------------------

print("transaction types in the source:")
base.groupBy("transaction_type").count().orderBy(col("count").desc()).show()

print("customer segments:")
base.select("customer_id", "customer_segment").distinct() \
    .groupBy("customer_segment").count().orderBy(col("count").desc()).show()

print("hour buckets (night = 22:00 to 05:00):")
base.groupBy("hour_bucket").count().orderBy("hour_bucket").show()

transaction types in the source:
+----------------+------+
|transaction_type| count|
+----------------+------+
|    POS_Purchase|769971|
|      E_Commerce|439209|
|    EFT_Transfer|396477|
|  ATM_Withdrawal|263663|
|    RTC_Transfer|220602|
|         Deposit|110078|
+----------------+------+

customer segments:
+----------------+-----+
|customer_segment|count|
+----------------+-----+
|     Mass_Retail|18348|
|        Affluent| 5665|
|         Premier| 2802|
| Private_Banking| 1496|
+----------------+-----+

hour buckets (night = 22:00 to 05:00):
+-----------+-------+
|hour_bucket|  count|
+-----------+-------+
|      Night|2200000|
+-----------+-------+



In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 1: roll up every transaction to the customer level
# we take the transaction-level table and collapse it down so each customer
# gets ONE row with all their metrics summarised
# ---------------------------------------------------------------------------
# what we are doing here:
#   - counting total transactions per customer
#   - summing money in (credits) and money out (debits)
#   - breaking down activity by channel (RTC, EFT, ATM)
#   - counting night and weekend transactions
#   - tracking how many different accounts and devices each customer uses
#   - pulling the latest Silver velocity numbers
# ---------------------------------------------------------------------------

customer_rollup = base.groupBy("customer_id").agg(

    # ---- who is this customer ----
    first("cust_full_name", ignorenulls=True).alias("customer_name"),
    first("customer_segment", ignorenulls=True).alias("segment"),
    first("cust_tenure_bucket", ignorenulls=True).alias("tenure_bucket"),

    # ---- overall activity ----
    count("transaction_id").alias("total_txns"),
    sum("transaction_amount").alias("total_amount"),
    round(avg("transaction_amount"), 2).alias("avg_txn_amount"),
    max("transaction_amount").alias("max_txn_amount"),

    # ---- money coming IN (credits from other accounts) ----
    # a transaction has a dest_account_id only when money is moving TO that account
    sum(when(col("dest_account_id").isNotNull(), col("transaction_amount")).otherwise(0)).alias("inflow_amount"),
    count(when(col("dest_account_id").isNotNull(), lit(1))).alias("inflow_count"),

    # ---- money going OUT (debits, purchases, withdrawals) ----
    sum(when(col("dest_account_id").isNull(), col("transaction_amount")).otherwise(0)).alias("outflow_amount"),

    # ---- RTC transfers (immediate interbank — favourite channel for mules) ----
    count(when(col("transaction_type") == "RTC_Transfer", lit(1))).alias("rtc_count"),
    sum(when(col("transaction_type") == "RTC_Transfer", col("transaction_amount")).otherwise(0)).alias("rtc_amount"),

    # ---- EFT transfers (slower but still used for layering) ----
    count(when(col("transaction_type") == "EFT_Transfer", lit(1))).alias("eft_count"),
    sum(when(col("transaction_type") == "EFT_Transfer", col("transaction_amount")).otherwise(0)).alias("eft_amount"),

    # ---- ATM cash withdrawals (cash extraction = high risk) ----
    count(when(col("transaction_type") == "ATM_Withdrawal", lit(1))).alias("atm_count"),
    sum(when(col("transaction_type") == "ATM_Withdrawal", col("transaction_amount")).otherwise(0)).alias("atm_amount"),

    # ---- night activity (22:00 to 05:00 — unusual for normal banking) ----
    count(when(col("hour_bucket") == "Night", lit(1))).alias("night_count"),

    # ---- weekend activity ----
    count(when(col("is_weekend") == 1, lit(1))).alias("weekend_count"),

    # ---- how many times did Silver flags trigger for this customer ----
    sum(when(col("high_velocity_flag") == 1, lit(1)).otherwise(0)).alias("velocity_flag_triggers"),
    sum(when(col("off_hours_flag") == 1, lit(1)).otherwise(0)).alias("off_hours_triggers"),

    # ---- how spread out is their activity ----
    countDistinct("transaction_date").alias("active_days"),
    countDistinct("source_account_id").alias("accounts_used"),
    countDistinct("device_id").alias("devices_used"),
    countDistinct("dest_account_id").alias("beneficiaries"),

    # ---- date range of their activity ----
    min("transaction_date").alias("first_txn_date"),
    max("transaction_date").alias("last_txn_date"),

    # ---- peak Silver velocity (the highest daily count we saw for this customer) ----
    max("customer_txn_count_30d").alias("peak_daily_txn_count"),
    max("customer_amount_sum_30d").alias("peak_daily_amount")
)

print(f"customers after rollup: {customer_rollup.count():,}")

customers after rollup: 28,311


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 2: device network risk
# a device used by many different customers is a fraud ring indicator
# we calculate how many other customers share each of this customer's devices
# ---------------------------------------------------------------------------
# approach:
#   1. for each device, count how many unique customers use it
#   2. for each customer, find the device with the MOST other customers
#   3. that number is our device risk signal
# ---------------------------------------------------------------------------

# step 2a: customers per device
device_sharing = base \
    .filter(col("device_id").isNotNull()) \
    .groupBy("device_id") \
    .agg(countDistinct("customer_id").alias("customers_on_device"))

# step 2b: for each customer, what is their riskiest device
customer_device_risk = base \
    .filter(col("device_id").isNotNull()) \
    .select("customer_id", "device_id").distinct() \
    .join(device_sharing, "device_id", "left") \
    .groupBy("customer_id") \
    .agg(
        max("customers_on_device").alias("max_others_on_device"),
        round(avg("customers_on_device"), 1).alias("avg_others_on_device"),
        count("device_id").alias("device_count")
    )

print(f"customers with device data: {customer_device_risk.count():,}")

customers with device data: 28,311


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 3: mule account detection flags
# mules follow predictable patterns:
#   - money arrives, then leaves quickly (pass-through)
#   - they send money to many different people (beneficiary spread)
#   - they prefer RTC for speed
#   - they operate at odd hours
# ---------------------------------------------------------------------------

# pass-through ratio: how much of what comes in goes straight out
# a ratio close to 1.0 means nearly everything that arrives gets sent away
customer_rollup = customer_rollup.withColumn(
    "pass_through_ratio",
    when(col("inflow_amount") > 0,
         round(col("outflow_amount") / col("inflow_amount"), 2))
    .otherwise(None)
)

# flag customers where money flows straight through
# we require: ratio between 0.8 and 1.2, at least R50,000 in flow, more than 5 credits
customer_rollup = customer_rollup.withColumn(
    "pass_through_flag",
    when(
        (col("pass_through_ratio").between(0.8, 1.2)) &
        (col("inflow_amount") > 50000) &
        (col("inflow_count") > 5),
        1
    ).otherwise(0)
)

# flag customers sending money to 10+ different beneficiaries
customer_rollup = customer_rollup.withColumn(
    "beneficiary_spread_flag",
    when(col("beneficiaries") > 10, 1).otherwise(0)
)

# flag heavy RTC users (20+ RTC transfers totalling over R100,000)
customer_rollup = customer_rollup.withColumn(
    "rtc_abuse_flag",
    when((col("rtc_count") > 20) & (col("rtc_amount") > 100000), 1).otherwise(0)
)

# night activity as a percentage of all activity
customer_rollup = customer_rollup.withColumn(
    "night_pct",
    when(col("total_txns") > 0,
         round(col("night_count") / col("total_txns"), 3))
    .otherwise(0)
)

# weekend activity as a percentage
customer_rollup = customer_rollup.withColumn(
    "weekend_pct",
    when(col("total_txns") > 0,
         round(col("weekend_count") / col("total_txns"), 3))
    .otherwise(0)
)

# velocity growth: is recent activity much higher than the 30-day average
# a spike suggests the account was dormant and suddenly became active
customer_rollup = customer_rollup.withColumn(
    "velocity_growth",
    when((col("peak_daily_txn_count") > 0) & (col("active_days") >= 7),
         round(col("total_txns") / col("active_days") /
               greatest(col("peak_daily_txn_count") / 30, lit(1)), 2))
    .otherwise(None)
)

print("mule flags built")

mule flags built


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 4: join device risk onto the customer rollup
# ---------------------------------------------------------------------------

customer_full = customer_rollup.join(customer_device_risk, "customer_id", "left")

# fill nulls for customers with no device data (they get zeros)
customer_full = customer_full \
    .fillna(0, ["max_others_on_device", "avg_others_on_device", "device_count"])

print(f"rows after device join: {customer_full.count():,}")

rows after device join: 28,311


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# CELL 7: FIU COMPOSITE RISK SCORE (revised — continuous percentile model)
# ---------------------------------------------------------------------------
# the old scoring used binary flags and coarse bins. this meant most
# customers got exactly the same score (25 points) and investigators
# could not tell who to look at first.
#
# the new approach is continuous: every customer gets a score from 0 to
# 100 based on how extreme they are compared to everyone else on each
# risk dimension. the more unusual your behaviour, the higher your score.
#
# 8 components, each weighted by investigative importance:
#   velocity risk        → 0 to 20 points
#   pass-through risk    → 0 to 25 points
#   rtc abuse            → 0 to 20 points
#   night activity       → 0 to 10 points
#   device network risk  → 0 to 10 points
#   beneficiary spread   → 0 to 10 points
#   velocity growth      → 0 to 10 points
#   atm extraction       → 0 to  5 points
#   total possible       → 110 (capped at 100)
#
# risk bands:
#   LOW       →  0 to 14  (normal banking, no action needed)
#   MEDIUM    → 15 to 29  (some patterns worth a second look)
#   HIGH      → 30 to 49  (multiple risk factors, prioritise)
#   CRITICAL  → 50 to 100 (strong mule/rtc/device indicators)
# ---------------------------------------------------------------------------

# ---------------------------------------------------------------------------
# STEP 1: figure out the maximum values across all customers
# we use these to scale each person's numbers into a 0-to-1 range
# someone with half the max gets half the points — no artificial cutoffs
# ---------------------------------------------------------------------------

max_vals = customer_full.agg(
    max("velocity_flag_triggers").alias("max_vel"),
    max("rtc_count").alias("max_rtc_n"),
    max("rtc_amount").alias("max_rtc_a"),
    max("inflow_amount").alias("max_inflow"),
    max("max_others_on_device").alias("max_others"),
    max("avg_others_on_device").alias("max_avg_others"),
    max("beneficiaries").alias("max_benef"),
    max("velocity_growth").alias("max_growth"),
    max("atm_amount").alias("max_atm")
).collect()[0]

# store them as python variables so we can use them in spark expressions
max_vel     = max_vals["max_vel"]
max_rtc_n   = max_vals["max_rtc_n"]
max_rtc_a   = max_vals["max_rtc_a"]
max_inflow  = max_vals["max_inflow"]
max_others  = max_vals["max_others"]
max_avg_oth = max_vals["max_avg_others"]
max_benef   = max_vals["max_benef"]
max_growth  = max_vals["max_growth"]
max_atm     = max_vals["max_atm"]

print(f"reference values for scaling:")
print(f"  max velocity triggers:     {max_vel}")
print(f"  max rtc count:             {max_rtc_n}")
print(f"  max rtc amount:            {max_rtc_a:,.0f}")
print(f"  max inflow amount:         {max_inflow:,.0f}")
print(f"  max others on device:      {max_others}")
print(f"  max avg others on device:  {max_avg_oth}")
print(f"  max beneficiaries:         {max_benef}")
print(f"  max velocity growth:       {max_growth}")
print(f"  max atm amount:            {max_atm:,.0f}")

# ---------------------------------------------------------------------------
# STEP 2: calculate each risk component
# every component scales the customer's raw number against the population max
# ---------------------------------------------------------------------------

# ---- velocity risk: how many times did silver flag high velocity ----
# a customer flagged 50 times gets twice the points of one flagged 25 times
customer_scored = customer_full.withColumn(
    "scr_velocity",
    when(lit(max_vel) > 0,
         round((col("velocity_flag_triggers") / lit(max_vel)) * 20, 1))
    .otherwise(0)
)

# ---- pass-through risk: how close is the ratio to 1.0 AND how big is the flow ----
# ratio of exactly 1.0 means every rand that came in went straight out
# we multiply by inflow size because R50 of pass-through is not suspicious
# R500,000 of pass-through is very suspicious
customer_scored = customer_scored \
    .withColumn(
        "ratio_closeness",
        when(col("pass_through_ratio").isNotNull(),
             1.0 - abs(col("pass_through_ratio") - 1.0))
        .otherwise(0)
    ) \
    .withColumn(
        "inflow_scale",
        least(col("inflow_amount") / lit(500000), lit(1.0))
    ) \
    .withColumn(
        "scr_pass_through",
        round(col("ratio_closeness") * col("inflow_scale") * 25, 1)
    )

# ---- rtc abuse: count and volume, equally weighted ----
# rtc is the mule's favourite channel because money moves instantly
# someone with 25 rtc transfers of R250k total gets about 10 points
# someone with 50+ transfers of R500k+ gets the full 20
customer_scored = customer_scored \
    .withColumn(
        "rtc_count_score",
        least(col("rtc_count") / lit(50), lit(1.0)) * 10
    ) \
    .withColumn(
        "rtc_amount_score",
        least(col("rtc_amount") / lit(500000), lit(1.0)) * 10
    ) \
    .withColumn(
        "scr_rtc",
        round(col("rtc_count_score") + col("rtc_amount_score"), 1)
    )

# ---- night activity: what percentage of transactions happen after hours ----
# 30% night activity = 3 points, 100% night activity = 10 points
customer_scored = customer_scored.withColumn(
    "scr_night",
    round(col("night_pct") * 10, 1)
)

# ---- device network risk: how many other people share your devices ----
# the worst device contributes 6 points, the average contributes 4
customer_scored = customer_scored \
    .withColumn(
        "scr_device_worst",
        when(lit(max_others) > 0,
             least(col("max_others_on_device") / lit(max_others), lit(1.0)) * 6)
        .otherwise(0)
    ) \
    .withColumn(
        "scr_device_avg",
        when(lit(max_avg_oth) > 0,
             least(col("avg_others_on_device") / lit(max_avg_oth), lit(1.0)) * 4)
        .otherwise(0)
    ) \
    .withColumn(
        "scr_device",
        round(col("scr_device_worst") + col("scr_device_avg"), 1)
    )

# ---- beneficiary spread: how many different people do you send money to ----
# 25 beneficiaries = 5 points, 50+ = 10 points
customer_scored = customer_scored.withColumn(
    "scr_beneficiaries",
    when(lit(max_benef) > 0,
         round(least(col("beneficiaries") / lit(max_benef), lit(1.0)) * 10, 1))
    .otherwise(0)
)

# ---- velocity growth: how much has activity spiked vs historical baseline ----
# 2.5x growth = 5 points, 5x+ = 10 points
customer_scored = customer_scored.withColumn(
    "scr_growth",
    round(least(col("velocity_growth") / lit(5), lit(1.0)) * 10, 1)
)

# ---- atm extraction: total cash taken out ----
# R100k = 2.5 points, R200k+ = 5 points
customer_scored = customer_scored.withColumn(
    "scr_atm",
    round(least(col("atm_amount") / lit(200000), lit(1.0)) * 5, 1)
)

# ---------------------------------------------------------------------------
# STEP 3: add up all the components and cap at 100
# ---------------------------------------------------------------------------

customer_scored = customer_scored \
    .withColumn(
        "fiu_risk_score_raw",
        col("scr_velocity") +
        col("scr_pass_through") +
        col("scr_rtc") +
        col("scr_night") +
        col("scr_device") +
        col("scr_beneficiaries") +
        col("scr_growth") +
        col("scr_atm")
    ) \
    .withColumn(
        "fiu_risk_score",
        round(least(col("fiu_risk_score_raw"), lit(100)), 0).cast("int")
    )

# ---------------------------------------------------------------------------
# STEP 4: assign human-readable risk bands
# ---------------------------------------------------------------------------

customer_scored = customer_scored.withColumn(
    "fiu_risk_band",
    when(col("fiu_risk_score") >= 50, "CRITICAL")
    .when(col("fiu_risk_score") >= 30, "HIGH")
    .when(col("fiu_risk_score") >= 15, "MEDIUM")
    .otherwise("LOW")
)

# ---------------------------------------------------------------------------
# STEP 5: check the distribution — it should look like a pyramid now
# ---------------------------------------------------------------------------

print("\nrisk band distribution:")
customer_scored.groupBy("fiu_risk_band").count().orderBy(col("count").desc()).show()

print("top 25 scores (should not see one number dominating):")
customer_scored.groupBy("fiu_risk_score").count() \
    .orderBy(col("fiu_risk_score").desc()).show(25)

print("score statistics:")
customer_scored.select("fiu_risk_score").describe().show()

reference values for scaling:
  max velocity triggers:     0
  max rtc count:             36
  max rtc amount:            2,166,798
  max inflow amount:         3,144,229
  max others on device:      20
  max avg others on device:  13.0
  max beneficiaries:         84
  max velocity growth:       1.27
  max atm amount:            104,830

risk band distribution:
+-------------+-----+
|fiu_risk_band|count|
+-------------+-----+
|       MEDIUM|15863|
|         HIGH|12438|
|     CRITICAL|    9|
|          LOW|    1|
+-------------+-----+

top 25 scores (should not see one number dominating):
+--------------+-----+
|fiu_risk_score|count|
+--------------+-----+
|            52|    2|
|            51|    2|
|            50|    5|
|            49|    1|
|            48|    7|
|            47|   18|
|            46|   19|
|            45|   27|
|            44|   77|
|            43|  157|
|            42|  223|
|            41|  319|
|            40|  427|
|            39|  638|
|            

In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# CELL 8: pick final columns and write the mart
# ---------------------------------------------------------------------------
# we select only the columns an investigator actually needs in power bi.
# the raw scoring components (scr_*) are included so investigators can
# see exactly what drove each customer's risk score.
# ---------------------------------------------------------------------------

target = f"{catalog}.{schema_name}.gold_fiu_customer_velocity_network"

final = customer_scored.select(

    # ---- identity ----
    "customer_id",
    "customer_name",
    "segment",
    "tenure_bucket",

    # ---- activity summary ----
    "total_txns",
    "total_amount",
    "avg_txn_amount",
    "max_txn_amount",
    "active_days",
    "first_txn_date",
    "last_txn_date",

    # ---- money flow ----
    "inflow_amount",
    "outflow_amount",
    "inflow_count",
    "pass_through_ratio",
    "beneficiaries",

    # ---- channel breakdown ----
    "rtc_count",
    "rtc_amount",
    "eft_count",
    "eft_amount",
    "atm_count",
    "atm_amount",

    # ---- behaviour ----
    "night_count",
    "night_pct",
    "weekend_count",
    "weekend_pct",
    "velocity_flag_triggers",
    "off_hours_triggers",

    # ---- velocity ----
    "peak_daily_txn_count",
    "peak_daily_amount",
    "velocity_growth",

    # ---- device network ----
    "devices_used",
    "max_others_on_device",
    "avg_others_on_device",
    "device_count",

    # ---- mule flags (still useful for quick filtering in power bi) ----
    "pass_through_flag",
    "beneficiary_spread_flag",
    "rtc_abuse_flag",

    # ---- scoring components (transparency: investigator can see what drove the score) ----
    "scr_velocity",
    "scr_pass_through",
    "scr_rtc",
    "scr_night",
    "scr_device",
    "scr_beneficiaries",
    "scr_growth",
    "scr_atm",

    # ---- final risk ----
    "fiu_risk_score",
    "fiu_risk_band",

    # ---- audit ----
    lit(batch_id).alias("batch_id"),
    current_timestamp().alias("built_at")
)

print(f"writing to {target}")
print(f"rows: {final.count():,}")
print(f"columns: {len(final.columns)}")

final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target)

print("done — table written")

writing to ubuntu_bank_fraud.gold.gold_fiu_customer_velocity_network
rows: 28,311
columns: 50
done — table written


In [0]:
# Databricks notebook source
# ---------------------------------------------------------------------------
# STEP 7: quick validation
# run a few checks to make sure everything looks right before we hand off
# ---------------------------------------------------------------------------

target = f"{catalog}.{schema_name}.gold_fiu_customer_velocity_network"
tbl = spark.table(target)

print("=" * 50)
print("VALIDATION")
print("=" * 50)

# check 1: we have one row per customer
total = tbl.count()
dupes = tbl.groupBy("customer_id").count().filter(col("count") > 1).count()
print(f"rows: {total:,}")
print(f"duplicate customers: {dupes}")

# check 2: no null risk scores
null_risk = tbl.filter(col("fiu_risk_score").isNull()).count()
print(f"null risk scores: {null_risk}")

# check 3: risk band breakdown
print("\nrisk bands:")
tbl.groupBy("fiu_risk_band").count().orderBy(col("count").desc()).show()

# check 4: top 10 riskiest customers — do they look like real mules
print("top 10 highest risk customers:")
tbl.select(
    "customer_id", "customer_name", "segment", "fiu_risk_score",
    "fiu_risk_band", "pass_through_flag", "rtc_abuse_flag",
    "total_txns", "inflow_amount", "max_others_on_device"
).orderBy(col("fiu_risk_score").desc()).show(10, truncate=False)

# check 5: pass-through flags
pt = tbl.filter(col("pass_through_flag") == 1).count()
print(f"\npass-through customers: {pt:,} ({pt/total*100:.1f}%)")

# check 6: RTC abuse flags
rtc = tbl.filter(col("rtc_abuse_flag") == 1).count()
print(f"rtc abuse customers: {rtc:,} ({rtc/total*100:.1f}%)")

print("\n" + "=" * 50)
print("validation complete")
print("=" * 50)

VALIDATION
rows: 28,311
duplicate customers: 0
null risk scores: 0

risk bands:
+-------------+-----+
|fiu_risk_band|count|
+-------------+-----+
|       MEDIUM|15863|
|         HIGH|12438|
|     CRITICAL|    9|
|          LOW|    1|
+-------------+-----+

top 10 highest risk customers:
+------------+-------------------+---------------+--------------+-------------+-----------------+--------------+----------+------------------+--------------------+
|customer_id |customer_name      |segment        |fiu_risk_score|fiu_risk_band|pass_through_flag|rtc_abuse_flag|total_txns|inflow_amount     |max_others_on_device|
+------------+-------------------+---------------+--------------+-------------+-----------------+--------------+----------+------------------+--------------------+
|CUST00004822|KGAUGELO MATLALA   |Affluent       |52            |CRITICAL     |0                |1             |234       |2661839.2600000007|19                  |
|CUST00026699|ELIZABETH COETZEE  |Affluent       |52    